In [30]:
import os
import collections
import numpy as np
from numpy.lib.format import open_memmap
from pathlib import Path
from tqdm import tqdm
import openwakeword
import openwakeword.data
import openwakeword.utils
import openwakeword.metrics
from openwakeword.utils import download_models
import scipy
import datasets
import matplotlib.pyplot as plt
import torch
from torch import nn
import IPython.display as ipd

In [28]:
cv_11 = datasets.load_dataset("mozilla-foundation/common_voice_11_0", "en", split="test", streaming=True)
cv_11 = cv_11.cast_column("audio", datasets.Audio(sampling_rate=16000, mono=True)) # convert to 16-khz
cv_11 = cv_11.cast_column("text", datasets.Value("string"))

# Convert and save clips (only first 5000)
limit = 5000
for i, example in tqdm(enumerate(cv_11), total=limit):
    if i >= limit:  # Stop if we've reached the limit
        break
    
    output = os.path.join("cv11_test_clips", example["path"][0:-4] + ".wav")
    os.makedirs(os.path.dirname(output), exist_ok=True)

    # Convert the audio array to 16-bit PCM format
    wav_data = (example["audio"]["array"] * 32767).astype(np.int16)
    
    # Save the audio clip as a .wav file
    scipy.io.wavfile.write(output, 16000, wav_data)

Reading metadata...: 16354it [00:00, 16400.60it/s]
100%|██████████| 5000/5000 [01:39<00:00, 50.16it/s]


In [33]:
download_models(
    target_directory="../models",
)

embedding_model.tflite: 100%|██████████| 1.33M/1.33M [00:00<00:00, 5.93MiB/s]
embedding_model.onnx: 100%|██████████| 1.33M/1.33M [00:00<00:00, 7.81MiB/s]
melspectrogram.tflite: 100%|██████████| 1.09M/1.09M [00:00<00:00, 8.66MiB/s]
melspectrogram.onnx: 100%|██████████| 1.09M/1.09M [00:00<00:00, 4.10MiB/s]
silero_vad.onnx: 100%|██████████| 1.81M/1.81M [00:00<00:00, 4.50MiB/s]
alexa_v0.1.tflite: 100%|██████████| 855k/855k [00:00<00:00, 8.32MiB/s]
alexa_v0.1.onnx: 100%|██████████| 854k/854k [00:00<00:00, 6.24MiB/s]
hey_mycroft_v0.1.tflite: 100%|██████████| 860k/860k [00:00<00:00, 5.13MiB/s]
hey_mycroft_v0.1.onnx: 100%|██████████| 858k/858k [00:00<00:00, 4.40MiB/s]
hey_jarvis_v0.1.tflite: 100%|██████████| 1.28M/1.28M [00:00<00:00, 7.56MiB/s]
hey_jarvis_v0.1.onnx: 100%|██████████| 1.27M/1.27M [00:00<00:00, 7.72MiB/s]
hey_rhasspy_v0.1.tflite: 100%|██████████| 416k/416k [00:00<00:00, 5.92MiB/s]
hey_rhasspy_v0.1.onnx: 100%|██████████| 204k/204k [00:00<00:00, 7.84MiB/s]
timer_v0.1.tflite: 100%|█

In [46]:
import os
#list current directory
os.listdir()
os.getcwd()
F = openwakeword.utils.AudioFeatures(melspec_model_path="/home/sebastian/Mine/dl/notebooks/melspectrogram.onnx", embedding_model_path="/home/sebastian/Mine/dl/notebooks/embedding_model.onnx")